# Phase 4 — Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

PROC_DIR = Path('../data/processed')
df = pd.read_csv(PROC_DIR / 'olist_merged_clean.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
])
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days


REFERENCE_DATE = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print('Reference date for recency/tenure calculations:', REFERENCE_DATE)
print('Loaded shape:', df.shape)

Reference date for recency/tenure calculations: 2018-12-09 23:58:00
Loaded shape: (42724, 43)


## Customer Features

### Total Orders, Total Spending, Avg Order Value, Avg Basket Size

- **Total Spending** uses `payment_value` (what the customer actually paid,
  including freight), not `price` alone.
- **Avg Basket Size** = average number of line items per order.

In [2]:
# Work order-level first (one row per order), then aggregate to customer-level

order_level = df.drop_duplicates('order_id')[
    ['order_id', 'customer_unique_id', 'payment_value', 'order_purchase_timestamp',
     'review_score', 'payment_type']
].copy()

print('Order-level shape:', order_level.shape)

items_per_order = df.groupby('order_id').size().rename('n_items')

order_level = order_level.merge(items_per_order, on='order_id')

print('Order-level shape after merging:', order_level.shape)
print(order_level.head())

customer_orders = order_level.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_spending=('payment_value', 'sum'),
    avg_basket_size=('n_items', 'mean'),
).reset_index()

customer_orders['avg_order_value'] = customer_orders['total_spending'] / customer_orders['total_orders']

customer_orders.head()

Order-level shape: (37618, 6)
Order-level shape after merging: (37618, 7)
                           order_id                customer_unique_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  7c396fd4830fd04220f754e42b4e5bff   
1  47770eb9100c2d0c44946d9cf07ec65d  3a653a41f6f9fc3d2a113cf8398680e8   
2  a4591c265e18cb1dcee52889e2d8acc3  80bb27c7c16e8f973207a5086ab329e2   
3  136cce7faa42fdb2cefd53fdc79a6098  36edbb3fb164b1f16485364b6fb04c73   
4  82566a660a982b15fb86e904c8d32918  e97109680b052ee858d93a539597bba7   

   payment_value order_purchase_timestamp  review_score payment_type  n_items  
0          38.71      2017-02-10 10:56:00           4.0      voucher        1  
1         179.12      2018-08-08 08:38:00           5.0  credit_card        1  
2         175.26      2017-09-07 21:57:00           4.0  credit_card        1  
3          65.95      2017-11-04 12:22:00           2.0  credit_card        1  
4          50.13      2018-07-06 10:06:00           5.0       boleto        1  


,customer_unique_id,total_orders,total_spending,avg_basket_size,avg_order_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,1.0,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,1.0,27.19
2,0004bd2a26a76fe21f786e4fbd80607f,1,166.98,1.0,166.98
3,0005e1862207bf6ccc02e4228effd9a0,1,150.12,1.0,150.12
4,000a5ad9c4601d2bbdd9ed765d5213b3,1,91.28,1.0,91.28


### Recency, Frequency, Monetary Value (RFM)

- **Recency** = days between the customer's last purchase and the reference date
- **Frequency** = total number of orders (same as Total Orders above)
- **Monetary Value** = total spending (same as Total Spending above)

RFM is deliberately built from the same base fields as §1a — that's expected;
RFM is a *lens* on the same numbers, not a separate computation.

In [3]:
rfm = order_level.groupby('customer_unique_id').agg(
    last_purchase_date=('order_purchase_timestamp', 'max'),
    first_purchase_date=('order_purchase_timestamp', 'min'),
).reset_index()

rfm['recency_days'] = (REFERENCE_DATE - rfm['last_purchase_date']).dt.days
rfm = rfm.merge(customer_orders[['customer_unique_id', 'total_orders', 'total_spending']],
                 on='customer_unique_id')
rfm = rfm.rename(columns={'total_orders': 'frequency', 'total_spending': 'monetary_value'})

rfm[['customer_unique_id', 'recency_days', 'frequency', 'monetary_value']].head()

,customer_unique_id,recency_days,frequency,monetary_value
0,0000366f3b9a7992bf8c76cfdf3221e2,65,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,157,1,27.19
2,0004bd2a26a76fe21f786e4fbd80607f,219,1,166.98
3,0005e1862207bf6ccc02e4228effd9a0,615,1,150.12
4,000a5ad9c4601d2bbdd9ed765d5213b3,396,1,91.28


### Customer Tenure, Days Since Last Purchase

- **Customer Tenure** = days since their *first* purchase (how long they've
  been a customer, as of the reference date)
- **Days Since Last Purchase** = identical to Recency above — the spec lists
  it separately, so it's kept as its own named column for clarity, but it's
  the same calculation.

In [4]:
rfm['customer_tenure_days'] = (REFERENCE_DATE - rfm['first_purchase_date']).dt.days
rfm['days_since_last_purchase'] = rfm['recency_days']  # same metric, kept as a named alias

rfm[['customer_unique_id', 'customer_tenure_days', 'days_since_last_purchase']].head()

,customer_unique_id,customer_tenure_days,days_since_last_purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,65,65
1,0000b849f77a49e4a4ce2b2a4ca5be3f,157,157
2,0004bd2a26a76fe21f786e4fbd80607f,219,219
3,0005e1862207bf6ccc02e4228effd9a0,615,615
4,000a5ad9c4601d2bbdd9ed765d5213b3,396,396


### Avg Review Rating, Preferred Payment Method

In [5]:
review_pref = order_level.groupby('customer_unique_id').agg(
    avg_review_rating=('review_score', 'mean'),
    preferred_payment_method=('payment_type', lambda x: x.mode().iat[0] if not x.mode().empty else np.nan),
).reset_index()

review_pref.head()

,customer_unique_id,avg_review_rating,preferred_payment_method
0,0000366f3b9a7992bf8c76cfdf3221e2,5.0,credit_card
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,credit_card
2,0004bd2a26a76fe21f786e4fbd80607f,4.0,credit_card
3,0005e1862207bf6ccc02e4228effd9a0,4.0,credit_card
4,000a5ad9c4601d2bbdd9ed765d5213b3,4.0,credit_card


In [6]:
# Assemble the full customer feature table
customer_features = (customer_orders
    .merge(rfm.drop(columns=['last_purchase_date', 'first_purchase_date']), on='customer_unique_id')
    .merge(review_pref, on='customer_unique_id'))

print('customer_features shape:', customer_features.shape)
customer_features.head()

customer_features shape: (36950, 12)


,customer_unique_id,total_orders,total_spending,avg_basket_size,avg_order_value,recency_days,frequency,monetary_value,customer_tenure_days,days_since_last_purchase,avg_review_rating,preferred_payment_method
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,1.0,141.90,65,1,141.90,65,65,5.0,credit_card
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,1.0,27.19,157,1,27.19,157,157,4.0,credit_card
2,0004bd2a26a76fe21f786e4fbd80607f,1,166.98,1.0,166.98,219,1,166.98,219,219,4.0,credit_card
3,0005e1862207bf6ccc02e4228effd9a0,1,150.12,1.0,150.12,615,1,150.12,615,615,4.0,credit_card
4,000a5ad9c4601d2bbdd9ed765d5213b3,1,91.28,1.0,91.28,396,1,91.28,396,396,4.0,credit_card


## Product Features

- **Product Popularity** = units sold (order-item count)
- **Product Rating** = average review score across orders containing the product
- **Category Popularity** = units sold at the category level, joined back onto every product in that category
- **Average Selling Price** = mean `price` for that product

In [7]:
product_review = df.dropna(subset=['review_score']).groupby('product_id')['review_score'].mean()

product_features = df.groupby('product_id').agg(
    product_category_name_english=('product_category_name_english', 'first'),
    product_popularity=('order_item_id', 'count'),       # units sold
    avg_selling_price=('price', 'mean'),
).reset_index()

product_features = product_features.merge(
    product_review.rename('product_rating'), on='product_id', how='left'
)

category_popularity = (df.groupby('product_category_name_english')
                          .size().rename('category_popularity').reset_index())
product_features = product_features.merge(category_popularity, on='product_category_name_english', how='left')

print('product_features shape:', product_features.shape)
product_features.sort_values('product_popularity', ascending=False).head()

product_features shape: (17802, 6)


,product_id,product_category_name_english,product_popularity,avg_selling_price,product_rating,category_popularity
12029,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,220,71.355000,4.032110,3097.0
10693,99a4788cb24856965c36a24e339b6058,bed_bath_table,194,88.106134,3.916667,4092.0
4659,422879e10f46682990de24d770e7f83d,garden_tools,143,56.379301,4.112676,1530.0
5892,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,138,114.847681,4.318841,2317.0
3965,389d119b48cf3043d311335e499d9c6b,garden_tools,134,55.514179,4.180451,1530.0


## Seller Features

- Seller Performance Score
- Average Delivery Time
- Average Review Score

In [8]:
seller_orders = df.dropna(subset=['delivery_time_days']).copy()
seller_orders['on_time'] = (seller_orders['delivery_delay_days'] <= 0).astype(int)

seller_features = df.groupby('seller_id').agg(
    seller_state=('seller_state', 'first'),
    n_orders=('order_id', 'nunique'),
).reset_index()

delivery_stats = seller_orders.groupby('seller_id').agg(
    avg_delivery_time_days=('delivery_time_days', 'mean'),
    on_time_rate=('on_time', 'mean'),
).reset_index()

review_stats = df.dropna(subset=['review_score']).groupby('seller_id')['review_score'].mean().rename('avg_review_score')

seller_features = (seller_features
    .merge(delivery_stats, on='seller_id', how='left')
    .merge(review_stats, on='seller_id', how='left'))

seller_features.head()

,seller_id,seller_state,n_orders,avg_delivery_time_days,on_time_rate,avg_review_score
0,0015a82c2db000af6aaaf3ae2ecb0532,SP,1,NaN,NaN,5.000000
1,001cca7ae9ae17fb1caed9dfb1094831,ES,90,171.687500,0.312500,3.980769
2,001e6ad469a905060d959994f1b41e4f,RJ,1,NaN,NaN,1.000000
3,002100f778ceb8431b7a1020ff7ab48f,SP,18,378.666667,0.000000,3.789474
4,004c9cd9d87a3c30c522c48c4fc07416,SP,65,179.222222,0.111111,4.338028


In [9]:
def min_max_norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

review_norm = min_max_norm(seller_features['avg_review_score'].fillna(seller_features['avg_review_score'].median()))
ontime_norm = min_max_norm(seller_features['on_time_rate'].fillna(seller_features['on_time_rate'].median()))
speed_norm = 1 - min_max_norm(seller_features['avg_delivery_time_days'].fillna(seller_features['avg_delivery_time_days'].median()))

seller_features['seller_performance_score'] = (
    0.5 * review_norm + 0.3 * ontime_norm + 0.2 * speed_norm
) * 100

print('seller_features shape:', seller_features.shape)
seller_features.sort_values('seller_performance_score', ascending=False).head()

seller_features shape: (2486, 7)


,seller_id,seller_state,n_orders,avg_delivery_time_days,on_time_rate,avg_review_score,seller_performance_score
1182,76ea5315acb3f5f7ade3cd261faf922a,SP,4,1.0,1.0,5.0,99.966387
1822,bb7a14e229b1c95156719c3d9552eeee,RJ,3,30.0,1.0,5.0,98.991597
1475,94ca168e8bcb407ab85c5da308863027,SP,2,31.0,1.0,5.0,98.957983
1186,777a0c55737f34ffeb78010f7542ab41,SP,1,58.0,1.0,5.0,98.050420
1571,9ed65337433adfbedefd23dd76eaa494,SP,1,58.0,1.0,5.0,98.050420


## Time-Based Features

### Purchase Month, Weekday, Hour, Seasonal Indicator

**Seasonal Purchase Indicator** flags Brazil's high-demand retail season
(Nov–Dec: Black Friday through Christmas/New Year), when order volume and
revenue are known from Phase 3's monthly revenue chart to spike.

In [10]:
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_weekday'] = df['order_purchase_timestamp'].dt.day_name()
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

df['seasonal_purchase_indicator'] = df['purchase_month'].isin([11, 12]).astype(int)

df[['order_purchase_timestamp', 'purchase_month', 'purchase_weekday',
    'purchase_hour', 'seasonal_purchase_indicator']].head()

,order_purchase_timestamp,purchase_month,purchase_weekday,purchase_hour,seasonal_purchase_indicator
0,2017-02-10 10:56:00,2,Friday,10,0
1,2018-08-08 08:38:00,8,Wednesday,8,0
2,2017-09-07 21:57:00,9,Thursday,21,0
3,2017-11-04 12:22:00,11,Saturday,12,1
4,2018-07-06 10:06:00,7,Friday,10,0


### Festival Sales Indicator (bonus)


In [11]:
def brazil_festival_dates(year):
    """Returns a dict of {festival_name: date} for a given year."""
    # Black Friday = last Friday of November
    nov_days = pd.date_range(f'{year}-11-01', f'{year}-11-30', freq='D')
    black_friday = nov_days[nov_days.day_name() == 'Friday'][-1]

    # Mother's Day (Brazil) = 2nd Sunday of May
    may_days = pd.date_range(f'{year}-05-01', f'{year}-05-31', freq='D')
    sundays = may_days[may_days.day_name() == 'Sunday']
    mothers_day = sundays[1] if len(sundays) > 1 else sundays[0]

    return {
        'valentines_br': pd.Timestamp(f'{year}-06-12'),   # Dia dos Namorados
        'black_friday': black_friday,
        'christmas': pd.Timestamp(f'{year}-12-25'),
        'mothers_day': mothers_day,
        'childrens_day': pd.Timestamp(f'{year}-10-12'),
    }

years = df['order_purchase_timestamp'].dt.year.unique()
festival_dates = []
for y in years:
    festival_dates.extend(brazil_festival_dates(int(y)).values())
festival_dates = pd.to_datetime(festival_dates)

WINDOW_DAYS = 3
purchase_dates = df['order_purchase_timestamp'].dt.normalize()

def is_near_festival(date):
    return any(abs((date - f).days) <= WINDOW_DAYS for f in festival_dates)

# Vectorized-ish: build a set of all flagged dates instead of looping per row
flagged_dates = set()
for f in festival_dates:
    for offset in range(-WINDOW_DAYS, WINDOW_DAYS + 1):
        flagged_dates.add((f + pd.Timedelta(days=offset)).normalize())

df['festival_sales_indicator'] = purchase_dates.isin(flagged_dates).astype(int)

print(f"{df['festival_sales_indicator'].mean()*100:.2f}% of order-items fall within a festival window")
df[['order_purchase_timestamp', 'festival_sales_indicator']].head()

6.56% of order-items fall within a festival window


,order_purchase_timestamp,festival_sales_indicator
0,2017-02-10 10:56:00,0
1,2018-08-08 08:38:00,0
2,2017-09-07 21:57:00,0
3,2017-11-04 12:22:00,0
4,2018-07-06 10:06:00,0


### Save engineered feature tables

In [12]:
customer_features.to_csv(PROC_DIR / 'customer_features.csv', index=False)
product_features.to_csv(PROC_DIR / 'product_features.csv', index=False)
seller_features.to_csv(PROC_DIR / 'seller_features.csv', index=False)
df.to_csv(PROC_DIR / 'olist_with_time_features.csv', index=False)

print('Saved:')
print(f'  customer_features.csv        {customer_features.shape}')
print(f'  product_features.csv         {product_features.shape}')
print(f'  seller_features.csv          {seller_features.shape}')
print(f'  olist_with_time_features.csv {df.shape}')

Saved:
  customer_features.csv        (36950, 12)
  product_features.csv         (17802, 6)
  seller_features.csv          (2486, 7)
  olist_with_time_features.csv (42724, 48)
